# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshithavalli1006-spec/FlyRank--AI-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

What this means in practice: The strongest observed signal is search position, while page length and content age show mixed relationships with impressions. The content team should prioritize pages with strong visibility and good search positions, but avoid relying on a single signal when deciding which pages to refresh. Provider differences should also be treated cautiously because many rows have missing provider information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 4.96 MiB/s, done.
Resolving deltas: 100% (161/161), done.


In [4]:
!ls -la flyrank-ml-internship-starter/data/raw/

total 6580
drwxr-xr-x 2 root root    4096 Sep  3 12:05 .
drwxr-xr-x 3 root root    4096 Sep  3 12:05 ..
-rw-r--r-- 1 root root 6727670 Sep  3 12:05 content_refresh_anonymized.csv


In [5]:
import pandas as pd

df = pd.read_csv(
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print("Shape:", df.shape)
print("Columns:", len(df.columns))

Shape: (30000, 44)
Columns: 44


In [6]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    "impressions_90d",
    "sessions_90d",
    "word_count",
    "avg_position"
]

# Target: declining pages
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

model_df = df[feature_cols + ["is_declining_label", "client_id"]].dropna()

X = model_df[feature_cols]
y = model_df["is_declining_label"]
groups = model_df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

Training rows: 17223
Test rows: 5078
Training positive rate: 0.5824188585031643
Test positive rate: 0.5214651437573848
Training clients: 25
Test clients: 7


Split design: I used a grouped-by-client split so that clients in the test set are unseen during training. This is a more honest test of generalization because pages from the same client are not shared across train and test. The split uses 25 clients for training and 7 unseen clients for testing.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# ---------- Recreate Week-4 baseline score ----------

def percentile_rank(s):
    return s.rank(pct=True)

def normalize(s):
    lo = s.min()
    hi = s.max()
    if hi == lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)

df["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"])
)

df["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"]
)

df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)

df["depth_gap_score"] = (
    (1 - percentile_rank(df["word_count"]))
    * df["visibility_score"]
)

df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)


# ---------- Train Random Forest ----------

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]


# ---------- Precision@50 ----------

def precision_at_50(y_true, scores):
    top_idx = np.argsort(scores)[::-1][:50]
    return y_true.iloc[top_idx].mean()

model_precision50 = precision_at_50(
    y_test,
    model_prob
)

baseline_scores_test = df.loc[X_test.index, "baseline_refresh_score"]

baseline_precision50 = precision_at_50(
    y_test,
    baseline_scores_test
)


comparison = pd.DataFrame({
    "method": ["Week-4 baseline", "Random Forest"],
    "precision_at_50": [
        baseline_precision50,
        model_precision50
    ]
})

print(comparison)
print("\nModel Precision@50:", round(model_precision50, 3))
print("Baseline Precision@50:", round(baseline_precision50, 3))

            method  precision_at_50
0  Week-4 baseline             0.36
1    Random Forest             0.64

Model Precision@50: 0.64
Baseline Precision@50: 0.36


Model vs baseline: The Random Forest achieved a Precision@50 of 0.64, compared with 0.36 for the Week-4 rule-based baseline. This means the model identified more truly declining pages among its top 50 recommendations on the unseen-client test set. The result is directional evidence that the trained model provides a stronger ranking signal than the simple baseline on this split.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# Error analysis for the Random Forest

test_results = df.loc[X_test.index, [
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "trend_direction"
]].copy()

test_results["actual"] = y_test
test_results["predicted_probability"] = model_prob
test_results["predicted"] = (model_prob >= 0.5).astype(int)

errors = test_results[
    test_results["actual"] != test_results["predicted"]
].copy()

print("Total test rows:", len(test_results))
print("Misclassified rows:", len(errors))
print("Error rate:", round(len(errors) / len(test_results), 3))

print("\nFalse positives:")
print(
    errors[errors["predicted"] == 1]
    .head(10)
    .to_string(index=False)
)

print("\nFalse negatives:")
print(
    errors[errors["predicted"] == 0]
    .head(10)
    .to_string(index=False)
)


Total test rows: 5078
Misclassified rows: 2162
Error rate: 0.426

False positives:
        client_id  impressions_90d  sessions_90d  avg_position trend_direction  actual  predicted_probability  predicted
client_8527a891e2              307             4          39.8          stable       0               0.749433          1
client_4e07408562             2426             9          30.0          stable       0               0.637796          1
client_f369cb89fc              371             5           5.4          stable       0               0.609357          1
client_f369cb89fc             2639             6           7.2              up       0               0.600966          1
client_8527a891e2               59             3           8.7              up       0               0.641897          1
client_4e07408562             1810             8           8.3              up       0               0.603619          1
client_4e07408562             1197             4          21.4        

Error analysis: The model has both false positives and false negatives, with 2,162 misclassified rows out of 5,078 test rows (42.6%). False positives include pages predicted as declining even though their observed trend is stable or up, while false negatives are declining pages that the model fails to identify. This shows that the model is useful for prioritization but should be treated as decision support rather than an automatic decision.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.